# Remote Inference — Cognitive Similarity

Runs TRIBE v2 inference on IBC stimuli and saves raw tensors to Google Drive.
All cells are safe to re-run — already-processed stimuli are skipped automatically.

**Runtime:** Google Colab Pro with A100 GPU.

## Cell 1 — Setup

In [ ]:
# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

# Clone/pull the git repo
import os
REPO_URL = "https://github.com/gdavid7/cogsimexperiment.git"
REPO_DIR = "/content/cogsimexperiment"
if not os.path.exists(REPO_DIR):
    !git clone {REPO_URL} {REPO_DIR}
else:
    !git -C {REPO_DIR} pull

# Install dependencies matching tribev2's requirements
# tribev2 requires: numpy==2.2.6, torch>=2.5.1,<2.7
# Strategy: Let tribev2 install its own dependencies, then install our extras

print("Installing tribev2 with its dependencies...")
!pip install -q git+https://github.com/facebookresearch/tribev2.git

print("Installing additional dependencies...")
!pip install -q nilearn scikit-learn hypothesis pandas transformers huggingface-hub jedi

print("\n" + "="*60)
print("✓ Installation complete!")
print("="*60)
print("\nNOTE: The runtime will now restart automatically.")
print("After restart, continue with Cell 1b.\n")

# Use IPython's restart mechanism instead of os.kill for cleaner restart
import IPython
IPython.Application.instance().kernel.do_shutdown(True)

## Cell 1b — Login (run after runtime restarts)

In [ ]:
# After the runtime restarts, re-run drive mount and login
from google.colab import drive
drive.mount('/content/drive')

import os
REPO_DIR = "/content/cogsimexperiment"
if not os.path.exists(REPO_DIR):
    !git clone https://github.com/gdavid7/cogsimexperiment.git {REPO_DIR}

# Verify installation
import numpy as np
import torch
print(f"NumPy version: {np.__version__}")
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

# HuggingFace login (for gated Llama 3.2 access used by TRIBE v2)
from huggingface_hub import login
login()  # will prompt for token

## Cell 2 — Load Models

In [ ]:
import sys
import os

# Define paths
REPO_DIR = "/content/cogsimexperiment"
CACHE_FOLDER = "/content/drive/MyDrive/cognitive-similarity-cache"

# Add repo to path
sys.path.insert(0, REPO_DIR)

# Import TRIBE v2
from tribev2 import TribeModel

# Create cache folder if it doesn't exist
os.makedirs(CACHE_FOLDER, exist_ok=True)

print("Loading cortical model...")
# Cortical model (default)
cortical_model = TribeModel.from_pretrained(
    "facebook/tribev2",
    cache_folder=CACHE_FOLDER,
    device="cuda"  # Explicitly use GPU
)

print("Loading subcortical model...")
# Subcortical model
subcortical_model = TribeModel.from_pretrained(
    "facebook/tribev2",
    cache_folder=CACHE_FOLDER,
    device="cuda",  # Explicitly use GPU
    config_update={
        "data.neuro.projection": {
            "name": "MaskProjector",
            "mask": "subcortical",
            "=replace=": True
        },
        "data.neuro.fwhm": 6.0
    }
)

print("✓ Models loaded successfully.")

## Cell 3 — Download IBC Stimuli and Build Manifest

In [ ]:
import json
import requests
from pathlib import Path
import sys

# Ensure repo is in path
sys.path.insert(0, REPO_DIR)

from cognitive_similarity.cache import ResponseCache
from cognitive_similarity.models import Stimulus

DRIVE_CACHE = Path(CACHE_FOLDER)
DRIVE_CACHE.mkdir(parents=True, exist_ok=True)

# IBC FaceBody stimuli base URL
IBC_BASE = "https://raw.githubusercontent.com/individual-brain-charting/public_protocols/master/FaceBody/stimuli"

# Define stimulus manifest entries
# Note: TRIBE v2 handles image inputs by converting them to 1-second static videos internally
STIMULUS_DEFINITIONS = [
    # Visual System stimuli (faces, places, bodies, written characters)
    {"stimulus_id": "face_01", "category": "face", "modality": "image", "github_url": f"{IBC_BASE}/adult/image_001.jpg"},
    {"stimulus_id": "face_02", "category": "face", "modality": "image", "github_url": f"{IBC_BASE}/adult/image_002.jpg"},
    {"stimulus_id": "place_01", "category": "place", "modality": "image", "github_url": f"{IBC_BASE}/places/image_001.jpg"},
    {"stimulus_id": "place_02", "category": "place", "modality": "image", "github_url": f"{IBC_BASE}/places/image_002.jpg"},
    {"stimulus_id": "body_01", "category": "body", "modality": "image", "github_url": f"{IBC_BASE}/body/image_001.jpg"},
    {"stimulus_id": "body_02", "category": "body", "modality": "image", "github_url": f"{IBC_BASE}/body/image_002.jpg"},
    # Add more as needed — this is a representative subset
]

# Download stimuli and compute content hashes
cache = ResponseCache(str(DRIVE_CACHE))
manifest = []
failed_downloads = []

for defn in STIMULUS_DEFINITIONS:
    local_path = DRIVE_CACHE / "stimuli" / defn["stimulus_id"]
    local_path.parent.mkdir(parents=True, exist_ok=True)

    if not local_path.exists():
        print(f"Downloading {defn['stimulus_id']} from {defn['github_url']}...")
        try:
            r = requests.get(defn["github_url"], timeout=30)
            r.raise_for_status()
            local_path.write_bytes(r.content)
            print(f"  ✓ Downloaded ({len(r.content)} bytes)")
        except Exception as e:
            print(f"  ✗ Failed: {e}")
            failed_downloads.append(defn["stimulus_id"])
            continue
    else:
        print(f"✓ {defn['stimulus_id']} already downloaded")

    # Compute content hash
    # Note: For images, we pass them as video_path - TRIBE v2 handles conversion internally
    stim = Stimulus(video_path=str(local_path), stimulus_id=defn["stimulus_id"])
    content_hash = cache._content_hash(stim)

    manifest.append({
        **defn,
        "local_path": str(local_path),
        "content_hash": content_hash,
        "tensor_dir": f"tensors/{content_hash}",
    })

# Save manifest
manifest_path = DRIVE_CACHE / "manifest.json"
manifest_path.write_text(json.dumps(manifest, indent=2))

print(f"\n=== Summary ===")
print(f"Manifest written: {len(manifest)} stimuli")
if failed_downloads:
    print(f"Failed downloads: {failed_downloads}")
    print("Note: You may need to verify IBC stimulus URLs or use alternative sources.")

## Cell 4 — Resume-from-Checkpoint Inference Loop

In [ ]:
import sys
sys.path.insert(0, REPO_DIR)

from cognitive_similarity.stimulus_runner import StimulusRunner
from cognitive_similarity.models import Stimulus
import numpy as np
from pathlib import Path

runner = StimulusRunner(cortical_model, subcortical_model)

processed = skipped = failed = 0

print(f"\n=== Processing {len(manifest)} stimuli ===\n")

for i, entry in enumerate(manifest, 1):
    stim = Stimulus(video_path=entry["local_path"], stimulus_id=entry["stimulus_id"])
    tensor_dir = DRIVE_CACHE / entry["tensor_dir"]
    cortical_path = tensor_dir / "raw_cortical.npy"
    subcortical_path = tensor_dir / "raw_subcortical.npy"

    # Skip if already cached
    if cortical_path.exists() and subcortical_path.exists():
        print(f"[{i}/{len(manifest)}] ✓ {entry['stimulus_id']:30s} (cached)")
        skipped += 1
        continue

    print(f"[{i}/{len(manifest)}] Processing {entry['stimulus_id']}...")
    
    try:
        # Run inference
        brain_response = runner.run(stim)

        # Verify shapes
        assert brain_response.cortical.shape[1] == 20484, f"Expected cortical shape (T, 20484), got {brain_response.cortical.shape}"
        assert brain_response.subcortical.shape[1] == 8802, f"Expected subcortical shape (T, 8802), got {brain_response.subcortical.shape}"

        # Save tensors
        tensor_dir.mkdir(parents=True, exist_ok=True)
        np.save(cortical_path, brain_response.cortical)
        np.save(subcortical_path, brain_response.subcortical)
        
        print(f"  ✓ Saved: cortical={brain_response.cortical.shape}, subcortical={brain_response.subcortical.shape}")
        processed += 1
        
    except Exception as e:
        print(f"  ✗ Failed: {e}")
        failed += 1
        continue

print(f"\n=== Summary ===")
print(f"Processed: {processed}")
print(f"Skipped (cached): {skipped}")
print(f"Failed: {failed}")
print(f"Total: {len(manifest)}")

## Cell 5 — Verify Cache

In [ ]:
import numpy as np
from pathlib import Path

print("=== Cache Verification ===\n")
tensors_dir = DRIVE_CACHE / "tensors"

for entry in manifest:
    tensor_dir = tensors_dir / entry["content_hash"]
    cortical_path = tensor_dir / "raw_cortical.npy"
    subcortical_path = tensor_dir / "raw_subcortical.npy"

    if cortical_path.exists():
        cortical = np.load(cortical_path)
        subcortical = np.load(subcortical_path) if subcortical_path.exists() else None
        sub_shape = subcortical.shape if subcortical is not None else "missing"
        print(f"\u2713 {entry['stimulus_id']:30s}  cortical={cortical.shape}  subcortical={sub_shape}")
    else:
        print(f"\u2717 {entry['stimulus_id']:30s}  NOT CACHED")